# unit07 レッスン: Transformer のファインチューニング

**題材** — unit05 と同じ中古品テキスト6クラス分類を、Transformer に渡せる入力と学習ループへ組み立てる。
TF-IDF + 線形モデルの実測 macro-F1 **0.9384**を比較の原点にし、深層モデルを使う価値を同じCVで判断する。

このレッスンでは torch / transformers や事前学習済み重みをダウンロードしない。
tokenization、attention mask、分類 head、学習率、更新順序を NumPy の極小モデルで実行し、
実務 API へそのまま読み替えられる形で学ぶ。

## このレッスンを終えると作れるようになるもの

1. サブワード、特殊 token、token ID、truncation の役割を説明して入力列を作れる
2. 可変長の例を動的 padding し、`input_ids` / `attention_mask` / `labels` の batch を作れる
3. warmup + linear decay の学習率をステップ番号から計算できる
4. pooled vector → 分類 head → loss → backward → 更新、という fine-tuning の順序を説明できる
5. train loss と valid metric から best epoch を選び、early stopping と重み復元を設計できる

所要の目安: **20〜25分**。このあと演習 `ex01`〜`ex04` が続く。

各概念を **見る → 予測する → 変える → 書く → チェック** の順で進む。
未記入セルは例外を出さず、チェックポイントが `[NG]` と修正の方向を返す。

In [ ]:
# STEP 1: このレッスンを終えると作れるようになるものの処理を実行し、出力を照合する
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

# unit07 は unit05 のテキストデータを共有する。ユニット直下 / リポジトリ直下の両方に対応。
DATA = Path("../unit05-text-classical-nlp/data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit05-text-classical-nlp/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

train = pd.read_csv(DATA / "train.csv")
print("train:", train.shape, "/ class数:", train["category"].nunique())
print("DATA =", DATA.resolve())


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def call_safely(fn, *args, **kwargs):
    # 未完成の関数を呼んでも notebook を止めない。
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数内で {type(e).__name__}: {e})")
        return None


def shape_safely(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None


def field_safely(mapping, key):
    try:
        return mapping[key]
    except Exception:
        return None


def item_safely(value, index):
    try:
        return value[index]
    except Exception:
        return None


print("セットアップ完了。ヘルパー: check / call_safely / shape_safely")

---
# 概念1 — サブワード tokenization

## ① なぜ: 未知語をゼロにしつつ、語彙を有限にする

単語単位の語彙では、新しいブランド名や型番 `XR-500B` が毎日増え、辞書にない語は全部 `[UNK]` になる。
文字単位なら未知語は減るが、系列が長くなり、attention の計算量が増える。
**サブワード**はその中間で、よく出る文字列をまとまりとして持ち、未知語を小さな既知パーツへ分解する。

BPE / WordPiece は、有限の語彙で新語へ対応する仕組み。検索語・商品名・SNSのように未知語が多い実務で効く。

## ② 解説: 文字列から model 入力 dict まで

BPE は文字から始め、学習済みの「隣接ペアを結合する規則」を順に適用する。
WordPiece は `##ing` のような印で「語の途中から続く断片」を表す。アルゴリズムは違うが、
最終的に**token のリストを整数ID列へ写す**点は同じだ。

| token / API | 何をするものか | 注意 |
|---|---|---|
| `[CLS]` | 文全体の分類に使う先頭 token | 分類 head はこの位置、または pooled vector を読む |
| `[SEP]` | 文や文対の終端・境界 | 文対では2文の間にも入る |
| `[PAD]` | batch の長さ合わせ | 情報ではないので attention mask は0 |
| `[UNK]` | 語彙で表せない断片 | 多すぎれば tokenizer / 正規化を疑う |
| `AutoTokenizer(...)` | 文字列を token ID と mask の dict へ変える実務 API | 事前学習モデルと同じ tokenizer を使う |

実 API の戻り値は `{"input_ids": ..., "attention_mask": ...}` の dict。
C# なら token 列を `Dictionary<string, int[]>` にまとめた DTO に近い。

> `from_pretrained(...)` は通常ネットワークから辞書を取得する。この教材では実行せず、固定の極小語彙を使う。

In [ ]:
# GOAL: BPE風のマージ規則で、未知の単語を既知のサブワードへまとめる過程を見る

def apply_merge(tokens, pair, merged):
    out = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair:
            out.append(merged)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out


pieces = list("playing")
rules = [
    (("p", "l"), "pl"),
    (("pl", "a"), "pla"),
    (("pla", "y"), "play"),
    (("i", "n"), "in"),
    (("in", "g"), "ing"),
]
print("開始:", pieces)
for pair, merged in rules:
    pieces = apply_merge(pieces, pair, merged)
    print(f"{pair} -> {merged:<4} :", pieces)
pieces = [pieces[0], "##" + pieces[1]]
print("最終サブワード:", pieces)

PAD_ID, UNK_ID, CLS_ID, SEP_ID = 0, 1, 2, 3
TOKEN_TO_ID = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3,
               "play": 4, "##ing": 5, "音楽": 6, "中古": 7}


def _encode_reference(tokens, max_length):
    body = [TOKEN_TO_ID.get(token, UNK_ID) for token in tokens]
    return [CLS_ID] + body[:max_length - 2] + [SEP_ID]


print("ID列:", _encode_reference(pieces, max_length=6))
print("特殊token込みの長さ:", len(_encode_reference(pieces, max_length=6)))

## ④ 予測: `max_length` を短くすると何が消える?

次のセルでは `中古 音楽 play ##ing` を、`max_length=6 / 5 / 4` で符号化する。

1. `[CLS]` と `[SEP]` を残すと、本文に使える長さはいくつ?
2. 末尾にカテゴリを決める語がある場合、先頭から切る truncation は何を失う?
3. `max_length` を半分にすると、self-attention の $L^2$ 部分はおよそ何分の1になる?

長さは精度・メモリ・レイテンシを同時に動かす設計値。分布を見て決める。

In [ ]:
# GOAL: 特殊tokenが長さ枠を使い、本文が truncation されることを見る

demo_tokens = ["中古", "音楽", "play", "##ing"]
for max_length in (6, 5, 4):
    ids = _encode_reference(demo_tokens, max_length=max_length)
    print(f"max_length={max_length}: {ids}  長さ={len(ids)}")

print("\n未知token:", _encode_reference(["未登録ブランド"], max_length=4))
print("語彙にない断片は UNK_ID=1。サブワード辞書が良ければ、実務では [UNK] をほぼゼロにできる。")
print("系列長 128 → 64 なら attention 行列は 128^2 → 64^2、約1/4。")

## ⑥ 書いてみる: token を特殊token付きID列へ変える

`encode_tokens(tokens, max_length)` を完成させよう。

- token は `TOKEN_TO_ID.get(token, UNK_ID)` でIDにする
- 本文は `max_length - 2` 個まで
- 先頭へ `CLS_ID`、末尾へ `SEP_ID`
- 戻り値は Python の `list[int]`

3〜4行で書ける。まだ padding はしない。長さ合わせは次の collate の責務に分ける。

In [ ]:
# STEP 4: ⑥ 書いてみる: token を特殊token付きID列へ変えるの処理を実行し、出力を照合する
def encode_tokens(tokens, max_length):
    # ここに書く(ヒント: 本文をIDへ写して max_length-2 まで切り、前後に特殊IDを足す)
    return None


encoded_a = call_safely(encode_tokens, ["play", "##ing"], 4)
print("encoded_a:", encoded_a)

In [ ]:
# STEP 5: ⑥ 書いてみる: token を特殊token付きID列へ変えるの処理を実行し、出力を照合する
# ===== チェックポイント A: token ID 化 =====
check("A-1 特殊token付きID", encoded_a, [2, 4, 5, 3],
      hint="[CLS] + play + ##ing + [SEP]。")
check("A-2 truncation", call_safely(encode_tokens, ["中古", "音楽", "play"], 4), [2, 7, 6, 3],
      hint="本文枠は max_length-2=2。末尾の play は切れる。")
check("A-3 未知token", call_safely(encode_tokens, ["未登録"], 3), [2, 1, 3],
      hint="dict.get(token, UNK_ID) で語彙外を1にする。")
check("A-4 戻り値は list", 1.0 if isinstance(encoded_a, list) else None, 1.0,
      hint="NumPy 配列ではなく、可変長の Python list を返す。padding は collate で行う。")

---
# 概念2 — Dataset / DataLoader / collate と attention mask

## ① なぜ: 文の長さは違うが、GPUは長方形の batch を欲しがる

文ごとの token 数は違う。一方、行列演算は `(batch, seq_len)` の長方形でまとめる方が速い。
`collate` は可変長の例を受け取り、その batch 内の最大長まで padding して1つの dict にする処理だ。

全データを常に `max_length=512` へ埋めると、短文の大半が `[PAD]` になり、計算時間とGPU課金を捨てる。
動的 padding と length grouping は、モデルを変えずにスループットを上げる実務の改善になる。

## ② 解説: 1件を返す Dataset、束ねる DataLoader

| PyTorch API | 何をするものか | C# アナロジー |
|---|---|---|
| `Dataset.__len__` | データ件数を返す | `IReadOnlyCollection<T>.Count` |
| `Dataset.__getitem__(i)` | 1件の token IDs と label を返す | `IReadOnlyList<T>[i]` |
| `DataLoader` | shuffle し、複数件を batch として順に返す | `IEnumerable<T[]>` |
| `collate_fn` | 可変長の1件群を長方形の配列へまとめる | `Func<List<T>, Batch>` |

collate の出力は通常3本:

- `input_ids`: `(batch, seq_len)` の整数。padding 位置は `PAD_ID`
- `attention_mask`: 同じ shape。本物token=1、padding=0
- `labels`: `(batch,)` のクラスID

モデル内部では mask を `(batch, 1, 1, seq_len)` へ広げ、padding の attention score を大きな負値にする。
softmax 後にはほぼ0になり、他のtokenから見えなくなる。unit06 の `[:, :, None]` と同じブロードキャストだ。

In [ ]:
# GOAL: batch 内最大長だけへ padding し、3本の shape を確認する

EXAMPLES = [
    {"input_ids": _encode_reference(["play", "##ing"], 6), "label": 0},
    {"input_ids": _encode_reference(["中古"], 6), "label": 1},
    {"input_ids": _encode_reference(["音楽", "中古", "play"], 6), "label": 2},
]

max_len_in_batch = max(len(ex["input_ids"]) for ex in EXAMPLES)
padded_rows = [ex["input_ids"] + [PAD_ID] * (max_len_in_batch - len(ex["input_ids"]))
               for ex in EXAMPLES]
input_ids_demo = np.array(padded_rows, dtype=int)
mask_demo = (input_ids_demo != PAD_ID).astype(int)
labels_demo = np.array([ex["label"] for ex in EXAMPLES], dtype=int)

print("元の長さ:", [len(ex["input_ids"]) for ex in EXAMPLES])
print("input_ids     :", input_ids_demo.shape)
print(input_ids_demo)
print("attention_mask:", mask_demo.shape)
print(mask_demo)
print("labels        :", labels_demo.shape, labels_demo)
print("padding token数:", int((input_ids_demo == PAD_ID).sum()))

## ④ 予測: 動的 padding と固定長、どちらがどれだけ無駄?

③ の3件の長さは `4, 3, 5`。次のセルで動的長5と固定長8を比べる。

1. padding token はそれぞれ何個?
2. attention mask `(batch, seq)` を `(batch, 1, 1, seq)` にするには、どこへ軸を足す?
3. padding score を `-1e9` にして softmax すると、確率は厳密に0? ほぼ0?

shape を紙に書いてから実行しよう。

In [ ]:
# GOAL: 動的 padding の削減量と、mask が attention 確率を0にする仕組みを見る

lengths = np.array([len(ex["input_ids"]) for ex in EXAMPLES])
dynamic_pad = int(np.sum(lengths.max() - lengths))
fixed_pad = int(np.sum(8 - lengths))
print("動的 padding(幅5):", dynamic_pad, "token")
print("固定 padding(幅8):", fixed_pad, "token")

# score の最後の軸が「参照先token」。mask はそこへ合わせる。
rng = np.random.default_rng(7)
scores = rng.normal(size=(3, 1, 1, input_ids_demo.shape[1]))
expanded_mask = mask_demo[:, None, None, :]
masked_scores = np.where(expanded_mask == 1, scores, -1e9)
exp = np.exp(masked_scores - masked_scores.max(axis=-1, keepdims=True))
attention_prob = exp / exp.sum(axis=-1, keepdims=True)

print("\nscores shape       :", scores.shape)
print("expanded_mask shape:", expanded_mask.shape)
print("attention_prob shape:", attention_prob.shape)
print("padding位置の最大確率:", float(attention_prob[expanded_mask == 0].max()))
print("大きな負値は softmax 後ほぼ0。padding を参照しなくなる。")

## ⑥ 書いてみる: 動的 padding の collate

`collate_dynamic(examples, pad_id=0)` を完成させよう。

1. batch 内の最大系列長を求める
2. 各 `input_ids` の右側へ `pad_id` を足す
3. NumPy 配列へし、`input_ids != pad_id` から mask を作る
4. labels も1次元配列へし、3本を dict で返す

6〜8行で書ける。まず3本の shape を揃えることを優先しよう。

In [ ]:
# STEP 8: ⑥ 書いてみる: 動的 padding の collateの処理を実行し、出力を照合する
def collate_dynamic(examples, pad_id=0):
    # ここに書く(ヒント: batch内最大長まで右paddingし、maskは input_ids != pad_id)
    return None


batch_b = call_safely(collate_dynamic, EXAMPLES, PAD_ID)
print("input_ids shape:", shape_safely(field_safely(batch_b, "input_ids")))

In [ ]:
# STEP 9: ⑥ 書いてみる: 動的 padding の collateの処理を実行し、出力を照合する
# ===== チェックポイント B: collate =====
_b_ids = field_safely(batch_b, "input_ids")
_b_mask = field_safely(batch_b, "attention_mask")
_b_labels = field_safely(batch_b, "labels")
check("B-1 input_ids shape", shape_safely(_b_ids), (3, 5),
      hint="batch内の最大長は5。全データ固定長ではない。")
check("B-2 attention_mask", _b_mask,
      [[1, 1, 1, 1, 0], [1, 1, 1, 0, 0], [1, 1, 1, 1, 1]],
      hint="本物token=1、右側のPADだけ0。")
check("B-3 labels", _b_labels, [0, 1, 2],
      hint="各exampleの label を入力順のまま1次元配列にする。")
_b_pad_count = int((_b_ids == PAD_ID).sum()) if _b_ids is not None else None
check("B-4 padding token数", _b_pad_count, 3,
      hint="長さ4,3,5を幅5へ揃えるので1+2+0。")

---
# 概念3 — warmup と learning-rate schedule

## ① なぜ: 事前学習済みの良い重みを、最初の数stepで壊さない

fine-tuning はランダム初期化からの学習と違い、すでに役立つ表現を持つ重みを少しだけ動かす。
最初から大きな学習率を掛けると、大きな勾配でその表現を壊す **catastrophic forgetting** が起きやすい。

warmup は学習率を0付近から徐々に上げ、その後 decay で小さくする。
Transformer の実務では `2e-5` など小さい peak と、総stepの5〜10%ほどの warmup がよく出発点になる。

## ② 解説: optimizer の更新回数を基準にする

| 用語 / API | 何をするものか | 注意 |
|---|---|---|
| warmup | 最初の数stepで0→peakへ増やす | batch数ではなく **optimizer.step 回数**で数える |
| linear decay | peak→0を直線で減らす | 終了stepで0になる |
| cosine decay | cosine 曲線でゆっくり始まり終盤に小さくする | schedule の選択も実験条件 |
| `torch.optim.lr_scheduler.LambdaLR` | step番号から倍率を返す関数でLRを制御 | `optimizer.step()` の後に `scheduler.step()` |
| `transformers.get_scheduler` | 名前・warmup数・総step数から scheduler を作る | 実 optimizer が必要 |

勾配累積を4回するなら、forward/backward は4回でも `optimizer.step()` は1回。
schedule も1回だけ進める。C# で言えば、micro-batch は集計途中で、commit した回数だけ状態番号を進める感覚だ。

In [ ]:
# GOAL: warmup + linear decay の学習率系列を表にする

def _linear_schedule(step, total_steps, warmup_steps, peak_lr):
    if step <= warmup_steps:
        return peak_lr * step / max(1, warmup_steps)
    progress_left = (total_steps - step) / max(1, total_steps - warmup_steps)
    return peak_lr * max(0.0, progress_left)


total_steps, warmup_steps, peak_lr = 10, 2, 2e-4
linear_lrs = [_linear_schedule(s, total_steps, warmup_steps, peak_lr)
              for s in range(total_steps + 1)]
print(f"{'step':>4} {'lr':>10}")
for step, lr in enumerate(linear_lrs):
    print(f"{step:>4} {lr:>10.7f}")
print("peak は step=2、その後は step=10 の0へ直線で下がる。")

## ④ 予測: linear と cosine はどこで違う?

次のセルでは warmup 後を linear / cosine の2通りにする。開始・peak・終了は同じ。

1. warmup 区間の値は一致する?
2. decay の中盤ではどちらが大きい?
3. scheduler を `optimizer.step()` より先に進めると、最初の学習率を使い損ねる可能性がある。更新順はどちらが先?

schedule は「どれが常に正しい」ではない。同じCVで比較する実験条件だ。

In [ ]:
# GOAL: 同じ始点・終点でも decay 曲線が異なることを見る

def cosine_schedule(step, total_steps, warmup_steps, peak_lr):
    if step <= warmup_steps:
        return peak_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return peak_lr * 0.5 * (1.0 + np.cos(np.pi * progress))


print(f"{'step':>4} {'linear':>10} {'cosine':>10}")
for step in (0, 1, 2, 4, 6, 8, 10):
    a = _linear_schedule(step, total_steps, warmup_steps, peak_lr)
    b = cosine_schedule(step, total_steps, warmup_steps, peak_lr)
    print(f"{step:>4} {a:>10.7f} {b:>10.7f}")

print("\n実務の更新順:")
print("loss.backward() → clip_grad_norm_ → optimizer.step() → scheduler.step()")
print("gradient accumulation 中は、最後のmicro-batchだけで step / scheduler を進める。")

## ⑥ 書いてみる: warmup + linear decay

`linear_warmup_decay(step, total_steps, warmup_steps, peak_lr)` を完成させよう。

- `step <= warmup_steps`: `peak_lr * step / warmup_steps`
- それ以降: `peak_lr * (total_steps - step) / (total_steps - warmup_steps)`
- 範囲外で負にならないよう `max(0.0, ...)`

4行ほど。step は0始まりとする。

In [ ]:
# STEP 12: ⑥ 書いてみる: warmup + linear decayの処理を実行し、出力を照合する
def linear_warmup_decay(step, total_steps, warmup_steps, peak_lr):
    # ここに書く(ヒント: warmup区間とdecay区間をifで分ける)
    return None


lr_c = call_safely(linear_warmup_decay, 6, 10, 2, 2e-4)
print("step6:", lr_c)

In [ ]:
# STEP 13: ⑥ 書いてみる: warmup + linear decayの処理を実行し、出力を照合する
# ===== チェックポイント C: learning-rate schedule =====
check("C-1 step0", call_safely(linear_warmup_decay, 0, 10, 2, 2e-4), 0.0,
      hint="warmup の開始は0。")
check("C-2 warmup終端", call_safely(linear_warmup_decay, 2, 10, 2, 2e-4), 0.0002,
      hint="step=warmup_steps で peak_lr。")
check("C-3 decay中盤", lr_c, 0.0001,
      hint="残りは4step、decay区間全体は8stepなので peak の半分。")
check("C-4 終了", call_safely(linear_warmup_decay, 10, 10, 2, 2e-4), 0.0,
      hint="total_steps で0。")

---
# 概念4 — 分類 head と fine-tuning loop

## ① なぜ: 「動いた」と「学べた」をローカルCPUで先に確認する

クラウドGPUを借りて最初に shape mismatch を見るのは、バグに課金している状態だ。
実務の安全な順序は、**CPUで1 batchの shape と loss → 数stepで loss が下がる → 本番データをGPU**。

Transformer 本体は token ごとの `(batch, seq, hidden)` を返し、分類 head が文1本の vector を
`(batch, classes)` の logits へ写す。今日は NumPy の埋め込み平均で本体を代用し、学習ループの配線を確認する。

## ② 解説: 実モデルでも更新順は同じ

`AutoModelForSequenceClassification` は「Transformer 本体 + dropout + 線形分類 head」をまとめた実務 API。
`from_pretrained` は事前学習重みを取得し、`from_config` はランダム初期化する。後者は仕組みのテストには使えるが、
**事前学習済み Transformer の精度を再現するものではない**。

| 順序 | PyTorch の操作 | 意味 |
|---|---|---|
| 1 | `optimizer.zero_grad()` | 前回の勾配を消す。累積時は更新直後だけ |
| 2 | `model.train(); outputs=model(**batch)` | dropout 等を学習モードにして forward |
| 3 | `loss.backward()` | 勾配を計算・加算 |
| 4 | `clip_grad_norm_` | 勾配爆発を抑える |
| 5 | `optimizer.step()` | 重みを更新 |
| 6 | `scheduler.step()` | 次の学習率へ進める |

検証時は `model.eval()` と `torch.no_grad()`。前者は dropout / BatchNorm の挙動を切り替え、
後者は勾配グラフを作らずメモリを節約する。役割が違うので両方必要。

GPUでは `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")` を1か所だけで決め、
model と batch を同じ device へ送る。AMP は活性値を低精度にしてメモリと時間を減らすが、GradScaler が必要になる。

In [ ]:
# GOAL: pooled vector → logits → softmax という分類headの入出力 shape を見る

def _softmax_reference(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


pooled_demo_d = np.array([[1.0, 0.0], [0.0, 1.0]])       # (batch=2, hidden=2)
weight_demo_d = np.array([[2.0, 0.0, -1.0],              # (hidden=2, classes=3)
                          [0.0, -1.0, 2.0]])
bias_demo_d = np.array([0.1, 0.0, 0.2])                  # (classes=3,)

logits_demo_d = pooled_demo_d @ weight_demo_d + bias_demo_d
probs_demo_d = _softmax_reference(logits_demo_d)
print("pooled:", pooled_demo_d.shape)
print("weight:", weight_demo_d.shape)
print("logits:", logits_demo_d.shape)
print("probs :", probs_demo_d.shape)
print(np.round(probs_demo_d, 4))
print("各行の合計:", probs_demo_d.sum(axis=1))

## ④ 予測: headだけ学習と、埋め込みも更新する fine-tuning

次のセルは極小モデルを40step学習する。比較するのは2条件。

- **head only**: token 埋め込みを凍結し、分類 head だけ更新
- **fine-tune all**: 埋め込みと head の両方を更新

1. どちらも loss は下がる?
2. 少数データで全層を大きく動かすと、なぜ過学習・事前表現の破壊が起きやすい?
3. 実務では「まず凍結して head → 足りなければ小さいLRで段階的に解凍」が安全なのはなぜ?

この NumPy モデルは Transformer 精度の代用品ではなく、更新の配線確認用。

In [ ]:
# GOAL: 数stepで loss が下がることをCPUで確認してから、本番GPUへ進む型を見る

toy_ids = np.array([
    [2, 4, 3, 0], [2, 4, 7, 3],
    [2, 5, 3, 0], [2, 5, 6, 3],
    [2, 6, 3, 0], [2, 6, 7, 3],
], dtype=int)
toy_mask = (toy_ids != 0).astype(float)
toy_labels = np.array([0, 0, 1, 1, 2, 2], dtype=int)


def train_toy(freeze_embeddings, steps=40):
    rng = np.random.default_rng(70)
    embedding = rng.normal(0, 0.15, size=(8, 4))
    weight = rng.normal(0, 0.10, size=(4, 3))
    bias = np.zeros(3)
    losses = []

    for step in range(steps + 1):
        token_vec = embedding[toy_ids]
        counts = toy_mask.sum(axis=1, keepdims=True)
        pooled = (token_vec * toy_mask[:, :, None]).sum(axis=1) / counts
        logits = pooled @ weight + bias
        probs = _softmax_reference(logits)
        loss = -np.log(probs[np.arange(len(toy_labels)), toy_labels] + 1e-12).mean()
        losses.append(float(loss))
        if step == steps:
            break

        dlogits = probs.copy()
        dlogits[np.arange(len(toy_labels)), toy_labels] -= 1
        dlogits /= len(toy_labels)
        dweight = pooled.T @ dlogits
        dbias = dlogits.sum(axis=0)
        dpooled = dlogits @ weight.T
        dtoken = dpooled[:, None, :] * toy_mask[:, :, None] / counts[:, :, None]
        dembedding = np.zeros_like(embedding)
        np.add.at(dembedding, toy_ids.reshape(-1), dtoken.reshape(-1, embedding.shape[1]))

        lr = _linear_schedule(step, steps, 4, 0.8)
        weight -= lr * dweight
        bias -= lr * dbias
        if not freeze_embeddings:
            embedding -= lr * dembedding
    return losses


loss_head = train_toy(freeze_embeddings=True)
loss_all = train_toy(freeze_embeddings=False)
assert loss_head[-1] < loss_head[0] and loss_all[-1] < loss_all[0]
print("head only    :", round(loss_head[0], 4), "→", round(loss_head[-1], 4))
print("fine-tune all:", round(loss_all[0], 4), "→", round(loss_all[-1], 4))
print("loss が下がる = 配線の最低確認。良い valid metric や事前学習の価値とは別問題。")

## ⑥ 書いてみる: 分類 head の forward

`classification_head(pooled, weight, bias)` を完成させよう。

1. `logits = pooled @ weight + bias`
2. 行ごとの最大値を引いて softmax を数値的に安定化
3. `np.exp` し、`axis=1, keepdims=True` の行合計で割る
4. `(batch, classes)` の確率を返す

4行ほど。まず3つの入力 shape を見て、行列積の内側が `hidden` で揃うことを確認する。

In [ ]:
# STEP 16: ⑥ 書いてみる: 分類 head の forwardの処理を実行し、出力を照合する
def classification_head(pooled, weight, bias):
    # ここに書く(ヒント: pooled @ weight + bias を行ごとにsoftmax)
    return None


probs_d = call_safely(classification_head, pooled_demo_d, weight_demo_d, bias_demo_d)
print("probs_d shape:", shape_safely(probs_d))

In [ ]:
# STEP 17: ⑥ 書いてみる: 分類 head の forwardの処理を実行し、出力を照合する
# ===== チェックポイント D: 分類 head =====
check("D-1 出力 shape", shape_safely(probs_d), (2, 3),
      hint="(batch, hidden) @ (hidden, classes) = (batch, classes)。")
_d_rowsum = probs_d.sum(axis=1) if probs_d is not None else None
check("D-2 各行の確率和", _d_rowsum, [1.0, 1.0],
      hint="softmax の分母は axis=1, keepdims=True の行合計。")
_d_pred = np.argmax(probs_d, axis=1) if probs_d is not None else None
check("D-3 予測クラス", _d_pred, [0, 2],
      hint="各行で最大確率の列番号。")
_d_first = float(probs_d[0, 0]) if probs_d is not None else None
check("D-4 1行目class0確率", _d_first, 0.849271578864049,
      hint="最大値を引くのは全logitを同じだけ平行移動するので、softmax確率は変わらない。")

---
# 概念5 — 過学習の診断、best epoch、fine-tune の判断

## ① なぜ: train loss が下がり続けても、本番性能は悪化する

数千件のラベルで巨大モデルを全層更新すると、訓練例を覚える方が先に進む。
train loss は学習を続けるほど下がりやすいが、valid macro-F1 は途中で頂点を迎えて落ちる。

提出や本番へ使うのは**最後の epoch ではなく、valid metric が最良だった重み**。
early stopping は時間節約であると同時に、過学習した重みを採用しないための仕組みだ。

## ② 解説: 保存・復元までが学習ループ

| 操作 / API | 何をするものか | 注意 |
|---|---|---|
| `model.state_dict()` | パラメータ名→tensor の辞書を得る | best 更新時にコピーして保存 |
| `torch.save(...)` | state_dict をファイルへ保存 | optimizer / scheduler 状態も再開には必要 |
| `model.load_state_dict(...)` | 保存した最良重みを復元 | 学習終了後に必ず戻す |
| patience | 改善なしを何epoch待つか | 小さ過ぎると指標の揺れで早く止まる |

fine-tuning を採用する判断は、unit05 と**同じ分割・同じ macro-F1**で行う。
TF-IDF の 0.9384 に対して改善がごく小さいなら、モデルサイズ、CPUレイテンシ、GPU学習費用まで含めると
古典モデルが勝つこともある。`from_config` のランダム初期化モデルが動いても、比較対象にはならない。

メモリ不足なら、まず `max_length` を見直す。attention は長さの二乗で効く。
次に batch を下げ、勾配累積で実効 batch を保ち、必要なら AMP を使う。

In [ ]:
# GOAL: train loss と valid metric の向きが途中から分かれることを見る

history = pd.DataFrame({
    "epoch": [1, 2, 3, 4, 5, 6],
    "train_loss": [1.20, 0.82, 0.56, 0.39, 0.27, 0.19],
    "valid_macro_f1": [0.620, 0.710, 0.760, 0.755, 0.742, 0.731],
})
best_row = history.loc[history["valid_macro_f1"].idxmax()]
print(history.to_string(index=False))
print("\nbest epoch:", int(best_row["epoch"]), "/ valid macro-F1:", best_row["valid_macro_f1"])
print("epoch3以降も train loss は下がるが valid は悪化。最後の重みを使ってはいけない。")

## ④ 予測: patience を変えると、いつ止まる?

valid metric は `[0.620, 0.710, 0.760, 0.755, 0.742, 0.731]`。

1. strict に「大きいときだけ改善」とし、patience=1 なら何epochで止まる?
2. patience=2 なら?
3. best epoch は patience を変えると変わる?
4. 指標が同値の plateau は改善として数える? ここでは数えない。

stop epoch と、復元する best epoch は別の値になる。

In [ ]:
# GOAL: patience は停止時点を変えるが、復元先は best metric のepochであることを見る

def _early_stop_reference(scores, patience):
    best_score = -np.inf
    best_epoch = -1
    bad_epochs = 0
    for epoch, score in enumerate(scores, start=1):
        if score > best_score:
            best_score, best_epoch, bad_epochs = score, epoch, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                return best_epoch, epoch
    return best_epoch, len(scores)


scores_e = history["valid_macro_f1"].tolist()
for patience in (1, 2, 3):
    best, stopped = _early_stop_reference(scores_e, patience)
    print(f"patience={patience}: best={best}, stop={stopped}")
print("停止後は best epoch の state_dict を load してから評価・提出する。")

## ⑥ 書いてみる: best epoch と stop epoch を返す

`find_best_and_stop(scores, patience)` を完成させよう。

- `best_score=-np.inf`, `best_epoch=-1`, `bad_epochs=0` から始める
- strict に `score > best_score` なら best を更新して `bad_epochs=0`
- 改善しなければ `bad_epochs += 1`
- `bad_epochs >= patience` で `(best_epoch, current_epoch)` を返す
- 最後まで止まらなければ `(best_epoch, len(scores))`

epoch は1始まり。7〜8行で書ける。

In [ ]:
# STEP 20: ⑥ 書いてみる: best epoch と stop epoch を返すの処理を実行し、出力を照合する
def find_best_and_stop(scores, patience):
    # ここに書く(ヒント: best更新ならbad=0、非改善ならbad+=1。epochは1始まり)
    return None


result_e = call_safely(find_best_and_stop, [0.60, 0.70, 0.69, 0.68], 2)
print("result_e:", result_e)

In [ ]:
# STEP 21: ⑥ 書いてみる: best epoch と stop epoch を返すの処理を実行し、出力を照合する
# ===== チェックポイント E: early stopping =====
check("E-1 patience=2", result_e, (2, 4),
      hint="epoch2がbest。epoch3,4の2回連続非改善でepoch4にstop。")
check("E-2 patience=1", call_safely(find_best_and_stop, [0.60, 0.70, 0.69, 0.68], 1), (2, 3),
      hint="最初の非改善epoch3でstop。復元先はepoch2。")
check("E-3 同値は改善でない", call_safely(find_best_and_stop, [0.80, 0.80, 0.80], 2), (1, 3),
      hint="条件は >= ではなく >。同値が2回続いてepoch3でstop。")

<!-- REAL_MODEL_SECTION_UNIT07 -->
## 実ライブラリで確認: 小さな BERT を1回更新する

ここまでの NumPy コードは Transformer の入出力と学習率を分解して見るための模型でした。実際の fine-tuning では、`BertConfig`（C# の設定オブジェクトに近い）から `BertForSequenceClassification`（分類 head 付き BERT）を作ります。

通常は事前学習済み重みを読み込みますが、このセルはオフライン検証できるよう極小構成をランダム初期化します。`forward → loss.backward() → optimizer.step()` という流れ自体は本番と同じです。

In [ ]:
# GOAL: Transformers の実モデルで forward・loss・backward・更新を1回通す
import torch
from transformers import BertConfig, BertForSequenceClassification

torch.manual_seed(7)
real_config = BertConfig(
    vocab_size=32,
    num_labels=2,
    max_position_embeddings=16,
    hidden_size=16,
    num_hidden_layers=1,
    num_attention_heads=2,
    intermediate_size=32,
    hidden_dropout_prob=0.0,
    attention_probs_dropout_prob=0.0,
)
real_bert = BertForSequenceClassification(real_config)
real_batch = {
    "input_ids": torch.tensor([[2, 4, 5, 3], [2, 6, 7, 3]]),
    "attention_mask": torch.ones((2, 4), dtype=torch.long),
    "labels": torch.tensor([0, 1]),
}
real_optimizer = torch.optim.AdamW(real_bert.parameters(), lr=1e-3)
real_optimizer.zero_grad()
real_output = real_bert(**real_batch)
real_output.loss.backward()
real_optimizer.step()
print("model:", type(real_bert).__name__)
print("logits shape:", tuple(real_output.logits.shape))
print("loss:", round(float(real_output.loss.detach()), 4))

---
## 振り返り(自己評価 + TIL)

以下に1〜2文ずつ、自分の言葉で書いてみよう。

**1. 今日学んだこと:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック:**

- 動的 padding が固定長 padding より安い理由を、attention の計算量と結びつけて説明できる?
- `optimizer.step()` と `scheduler.step()` の順序、勾配累積時に進める回数を説明できる?
- train loss が下がり続けているのに epoch3 の重みを採用する理由を説明できる?

> (ここに書く)

---
## まとめ

| 概念 | 一言でいうと |
|---|---|
| サブワード | 単語と文字の中間。未知語を減らし、語彙と系列長を有限にする |
| 特殊token | `[CLS]` / `[SEP]` / `[PAD]` / `[UNK]`。本文に使える長さは `max_length-2` |
| Dataset / DataLoader | 1件を返す部品と、shuffle・batch化して列挙する部品 |
| 動的 padding | batch 内最大長だけにそろえ、無駄 token と attention 計算を減らす |
| attention mask | `(batch, seq)` → `(batch,1,1,seq)`。padding score を大負値にして確率0へ |
| warmup + decay | 良い初期重みを最初に壊さず、終盤は小さい更新で詰める |
| fine-tuning loop | zero_grad → forward → backward → clip → optimizer → scheduler |
| 勾配累積 / AMP | 実効batchを保ってメモリを減らす / 低精度で活性値メモリと時間を減らす |
| best epoch | valid metric が最大の state_dict を保存・復元。最後のepochとは限らない |
| 採用判断 | unit05 の macro-F1 0.9384 と同じCVで比べ、精度差をコストとレイテンシに照らす |

### この先どこで使うか

- **unit10** — モデルサイズ、batch、系列長、CPU/GPUで推論コストを見積もる。今日の `max_length` が二乗で効く。
- **unit11** — attention mask、teacher forcing、生成時の token 列で同じ shape 規律を使う。
- **実務** — CPUで1 batch → 数step → GPU本番、best weight 復元、古典ベースラインとの費用対効果比較をそのまま使う。

**次は演習 `ex01_subword_tokenizer` へ進もう。lesson.ipynb を見ながらで OK。**

| 演習 | 内容 |
|---|---|
| `ex01_subword_tokenizer` | BPE風マージ、特殊token、ID化 |
| `ex02_collate_and_mask` | 動的 padding、attention mask、length grouping |
| `ex03_lr_schedule` | warmup + linear / cosine schedule |
| `ex04_capstone` | 極小モデルの学習ループ、best epoch、重み復元 |